In [1]:
import pandas as pd 
import numpy as np 
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

In [ ]:
class CF(object):
    """Collaborative Filtering Class for recommendation systems."""
    
    def __init__(self, Y_data, k, dist_func=cosine_similarity, CFmode=1):
        """
        Initialize the CF model.
        
        Parameters:
        - Y_data: Rating data matrix with rows as [user_id, item_id, rating].
        - k: Number of neighbors to consider for recommendations.
        - dist_func: Function to compute similarity, default is cosine similarity.
        - CFmode: Type of collaborative filtering (1 for user-user, 0 for item-item).
        """
        self.CFmode = CFmode  # Type of CF: user-user (1) or item-item (0)
        self.Y_data = Y_data if CFmode else Y_data[:, [1, 0, 2]]
        self.k = k  # Number of neighbors
        self.dist_func = dist_func
        self.Ybar_data = None
        # Number of users and items (IDs assumed to start from 0)
        self.n_users = int(np.max(self.Y_data[:, 0])) + 1 
        self.n_items = int(np.max(self.Y_data[:, 1])) + 1
    
    def normalize_matrix(self):
        """
        Normalize the ratings in Y_data by subtracting each user’s mean rating.
        
        - For each user, calculates the mean rating and subtracts it from 
          their individual ratings.
        - The normalized data is stored in Ybar_data.
        - Also constructs a sparse matrix Ybar for efficient computation.
        """
        users = self.Y_data[:, 0]  # All user IDs (first column of Y_data)
        self.Ybar_data = self.Y_data.copy()
        self.mu = np.zeros((self.n_users,))  # Mean ratings per user
        
        for n in range(self.n_users):
            # Indices of all ratings by user n
            ids = np.where(users == n)[0]
            # Item IDs and ratings given by user n
            item_ids = self.Y_data[ids, 1] 
            ratings = self.Y_data[ids, 2]
            # Mean rating by user n
            m = np.mean(ratings) 
            if np.isnan(m):
                m = 0  # Handle cases with no ratings
            self.mu[n] = m
            # Normalize ratings by subtracting the mean
            self.Ybar_data[ids, 2] = ratings - self.mu[n]

        # Create a sparse matrix with normalized ratings for efficiency
        self.Ybar = sparse.coo_matrix((self.Ybar_data[:, 2],
            (self.Ybar_data[:, 1], self.Ybar_data[:, 0])), shape=(self.n_items, self.n_users))
        self.Ybar = self.Ybar.tocsr()

    def similarity(self):
        """
        Calculate the similarity matrix using the specified distance function.
        
        - Adds a small epsilon to avoid issues with zero similarities.
        - Stores the similarity matrix in S.
        """
        eps = 1e-6
        self.S = self.dist_func(self.Ybar.T, self.Ybar.T) + eps
    
    def refresh(self):
        """
        Refresh the normalized data and similarity matrix.
        
        - Should be called after adding new ratings to re-compute normalization 
          and similarity matrices.
        """
        self.normalize_matrix()
        self.similarity()
        
    def fit(self):
        """
        Prepare the CF model by normalizing data and computing similarity matrix.
        
        - Calls the refresh function to perform both tasks.
        """
        self.refresh()
    
    def __pred(self, u, i, normalized=1):
        """ 
        Predict the rating for a user-item pair.
        
        Parameters:
        - u: User ID for whom we want to predict the rating.
        - i: Item ID to be predicted.
        - normalized: If 1, returns the normalized predicted rating, else adds the user mean.
        
        Steps:
        - Identifies users who rated item i.
        - Computes similarity between user u and these users.
        - Takes the k most similar users and their ratings.
        - Returns a weighted sum of these ratings based on similarity.
        """
        # Find all users who rated item i
        ids = np.where(self.Y_data[:, 1] == i)[0]
        # Users who rated item i
        users_rated_i = self.Y_data[ids, 0]
        # Similarity between user u and these users
        sim = self.S[u, users_rated_i]
        # Get the k most similar users
        a = np.argsort(sim)[-self.k:] 
        nearest_s = sim[a]
        # Ratings by these users for item i
        r = self.Ybar[i, users_rated_i[a]]
        
        if normalized:
            # Add a small value to avoid division by zero
            return (r * nearest_s).sum() / (np.abs(nearest_s).sum() + 1e-8)

        return (r * nearest_s).sum() / (np.abs(nearest_s).sum() + 1e-8) + self.mu[u]
    
    def pred(self, u, i, normalized=1):
        """ 
        Predict the rating of a user for an item.
        
        Parameters:
        - u: User ID.
        - i: Item ID.
        - normalized: If 1, returns the normalized predicted rating, else adds the user mean.
        
        Returns the predicted rating based on the collaborative filtering mode.
        """
        if self.CFmode:
            return self.__pred(u, i, normalized)
        return self.__pred(i, u, normalized)
            
    def recommend(self, u):
        """
        Generate item recommendations for a user.
        
        Parameters:
        - u: User ID.
        
        Returns:
        - List of item IDs recommended for user u.
        
        - Finds items not rated by the user and predicts ratings for them.
        - Recommends items with predicted rating > 0 (indicating likely interest).
        """
        ids = np.where(self.Y_data[:, 0] == u)[0]
        items_rated_by_u = self.Y_data[ids, 1].tolist()              
        recommended_items = []
        
        for i in range(self.n_items):
            if i not in items_rated_by_u:
                rating = self.__pred(u, i)
                if rating > 0: 
                    recommended_items.append(i)
        
        return recommended_items[:5]  # Return top 5 recommended items
    
    def print_recommendation(self):
        """
        Print recommended items for each user.
        
        - Iterates over each user and prints items recommended based on CF mode.
        """
        print("Recommendation:")
        for u in range(self.n_users):
            recommended_items = self.recommend(u)
            if self.CFmode:
                print(f'Recommend item(s): {recommended_items} for user {u}')
            else: 
                print(f'Recommend item {u} for user(s): {recommended_items}')
        
        print("***************************************************************************")


In [3]:
# data file 
r_cols = ['UserID', 'MovieID', 'Rating']
ratings_training = pd.read_csv('training_data.csv', sep = ',', encoding='latin-1')
ratings_testing = pd.read_csv('testing_data.csv', sep = ',', encoding='latin-1')

ratings_training = ratings_training.values
ratings_testing = ratings_testing.values

# indices start from 0
ratings_training[:, :2] -= 1
ratings_testing[:, :2] -= 1

In [4]:
rs = CF(ratings_training, k = 30, CFmode=1)
rs.fit()

rs.print_recommendation()

n_tests = ratings_testing.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(ratings_testing[n, 0], ratings_testing[n, 1], normalized = 0)
    SE += (pred - ratings_testing[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print('User-user CF, RMSE =', RMSE)

Recommendation:
Recommend item(s): [5, 9, 11, 13, 22] for user 0
Recommend item(s): [5, 6, 7, 8, 10] for user 1
Recommend item(s): [0, 3, 5, 6, 7] for user 2
Recommend item(s): [0, 5, 6, 7, 8] for user 3
Recommend item(s): [0, 6, 7, 9, 10] for user 4
Recommend item(s): [3, 5, 9, 10, 13] for user 5
Recommend item(s): [0, 5, 6, 7, 9] for user 6
Recommend item(s): [0, 6, 7, 9, 11] for user 7
Recommend item(s): [4, 7, 8, 9, 11] for user 8
Recommend item(s): [0, 1, 4, 6, 7] for user 9
Recommend item(s): [0, 8, 9, 11, 12] for user 10
Recommend item(s): [0, 2, 6, 7, 9] for user 11
Recommend item(s): [5, 10, 13, 14, 17] for user 12
Recommend item(s): [0, 6, 7, 9, 10] for user 13
Recommend item(s): [0, 5, 6, 7, 8] for user 14
Recommend item(s): [6, 7, 8, 9, 18] for user 15
Recommend item(s): [5, 9, 11, 13, 14] for user 16
Recommend item(s): [0, 6, 9, 10, 11] for user 17
Recommend item(s): [0, 4, 6, 8, 9] for user 18
Recommend item(s): [0, 6, 7, 8, 9] for user 19
Recommend item(s): [5, 7, 9, 10,

In [5]:
rs = CF(ratings_training, k = 30, CFmode = 0)
rs.fit()

rs.print_recommendation()

n_tests = ratings_testing.shape[0]
SE = 0 # squared error
for n in range(n_tests):
    pred = rs.pred(ratings_testing[n, 0], ratings_testing[n, 1], normalized = 0)
    SE += (pred - ratings_testing[n, 2])**2 

RMSE = np.sqrt(SE/n_tests)
print('Item-item CF, RMSE =', RMSE)

C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Recommendation:
Recommend item 0 for user(s): [3, 6, 7, 8, 9]
Recommend item 1 for user(s): [3, 6, 7, 8, 9]
Recommend item 2 for user(s): [3, 6, 7, 9, 10]
Recommend item 3 for user(s): [3, 7, 8, 9, 13]
Recommend item 4 for user(s): [1, 3, 6, 9, 11]
Recommend item 5 for user(s): [0, 1, 3, 6, 9]
Recommend item 6 for user(s): [1, 3, 4, 6, 7]
Recommend item 7 for user(s): [1, 2, 3, 6, 7]
Recommend item 8 for user(s): [1, 3, 7, 8, 10]
Recommend item 9 for user(s): [0, 2, 3, 6, 9]
Recommend item 10 for user(s): [1, 8, 11, 13, 17]
Recommend item 11 for user(s): [3, 7, 8, 11, 18]
Recommend item 12 for user(s): [3, 6, 8, 9, 11]
Recommend item 13 for user(s): [0, 2, 3, 5, 6]
Recommend item 14 for user(s): [1, 2, 6, 7, 8]
Recommend item 15 for user(s): [2, 3, 6, 7, 8]
Recommend item 16 for user(s): [6, 7, 8, 9, 11]
Recommend item 17 for user(s): [1, 3, 6, 8, 9]
Recommend item 18 for user(s): [1, 3, 6, 7, 8]
Recommend item 19 for user(s): [0, 1, 6, 9, 13]
Recommend item 20 for user(s): [2, 3, 6, 7